# Phase 2 — SQL Analysis (SQLite)

**Input:** Clean CSVs from Phase 1 (`data/clean/`)  
**Goal:** Load the data into a relational database and answer analytical questions with SQL.

**Output:** `data/fitness.db` — a portable SQLite database with three tables.

---
### Notebook structure
1. Setup & Connect
2. Load Data
3. Verify & Inspect Schema
4. Analysis Queries

## 1. Setup & Connect

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.1f}".format)

In [2]:
CLEAN_DIR = Path("../data/clean")
DB_PATH   = Path("../data/fitness.db")

# Close any lingering connection before deleting (Windows locks open files)
try:
    conn.close()
except NameError:
    pass

if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
print(f"Connected: {DB_PATH.resolve()}")
print(f"DB existed before: {not DB_PATH.exists()}")

Connected: C:\projects\FitnessData\data\fitness.db
DB existed before: False


## 2. Load Data

We let pandas infer and create the table structure directly from the DataFrames.
The intended relational schema is:

```sql
workouts  (workout_id PK, date, workout_name, duration_min,
           total_sets, total_volume_kg, peak_estimated_1rm, unique_exercises)

exercises (exercise_name PK, total_sets, first_seen, last_seen)

sets      (id PK AUTOINCREMENT, workout_id FK, date, exercise_name FK,
           set_order, weight_kg, reps, rpe, volume_kg, estimated_1rm)
```

In [3]:
workouts = pd.read_csv(CLEAN_DIR / "workout_summary.csv", parse_dates=["date"])
sets     = pd.read_csv(CLEAN_DIR / "clean_sets.csv",      parse_dates=["date"])

print(f"workouts : {len(workouts):,} rows | columns: {list(workouts.columns)}")
print(f"sets     : {len(sets):,} rows")

workouts : 849 rows | columns: ['workout_id', 'date', 'workout_name', 'total_sets', 'total_volume_kg', 'peak_estimated_1rm', 'unique_exercises', 'duration_min']
sets     : 18,830 rows


In [4]:
# Derive exercises lookup table from sets
exercises = (
    sets
    .groupby("exercise_name", as_index=False)
    .agg(
        total_sets = ("set_order", "count"),
        first_seen = ("date",      "min"),
        last_seen  = ("date",      "max"),
    )
)
print(f"exercises: {len(exercises):,} unique exercises")

exercises: 120 unique exercises


In [5]:
SETS_COLUMNS = [
    "workout_id", "date", "exercise_name", "set_order",
    "weight_kg", "reps", "rpe", "volume_kg", "estimated_1rm",
]

# if_exists='replace': pandas drops and recreates the table from the DataFrame —
# avoids any schema mismatch between what we expect and what exists on disk
workouts.to_sql("workouts",   conn, if_exists="replace", index=False)
exercises.to_sql("exercises", conn, if_exists="replace", index=False)
sets[SETS_COLUMNS].to_sql("sets", conn, if_exists="replace", index=False)

conn.commit()
print("Data loaded.")

Data loaded.


## 3. Verify & Inspect Schema

In [6]:
# Row counts must match the source CSVs
for table in ["workouts", "exercises", "sets"]:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn).iloc[0, 0]
    print(f"{table:<12}: {count:,} rows")

workouts    : 849 rows
exercises   : 120 rows
sets        : 18,830 rows


In [7]:
# Confirm actual column names in the workouts table
pd.read_sql("PRAGMA table_info(workouts)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,workout_id,INTEGER,0,None,0
1,1,date,TIMESTAMP,0,None,0
2,2,workout_name,TEXT,0,None,0
3,3,total_sets,INTEGER,0,None,0
4,4,total_volume_kg,REAL,0,None,0
5,5,peak_estimated_1rm,REAL,0,None,0
6,6,unique_exercises,INTEGER,0,None,0
7,7,duration_min,REAL,0,None,0


## 4. Analysis Queries

Each query answers a concrete question about the training data.

### Q1 — Personal Records: Top Estimated 1RM per Exercise

Which exercises have the highest all-time estimated 1RM?  
The subquery finds the maximum per exercise; the join retrieves the full set details.

In [8]:
q1 = """
SELECT
    s.exercise_name,
    ROUND(s.estimated_1rm, 1)  AS personal_record_kg,
    s.weight_kg,
    s.reps,
    DATE(s.date)               AS date
FROM sets s
INNER JOIN (
    SELECT exercise_name, MAX(estimated_1rm) AS max_1rm
    FROM sets
    GROUP BY exercise_name
) pr ON s.exercise_name = pr.exercise_name
     AND s.estimated_1rm = pr.max_1rm
ORDER BY personal_record_kg DESC
LIMIT 15;
"""
pd.read_sql(q1, conn)

,exercise_name,personal_record_kg,weight_kg,reps,date
0,Leg Press,383.3,250.0,16,2020-08-05
1,Hip Adductor (Machine),207.6,117.5,23,2024-08-16
2,Seated Leg Press (Machine),183.3,137.5,10,2020-07-15
3,Leg Extension (Machine),164.7,95.0,22,2022-04-11
4,Triceps Extension (Machine),164.0,102.5,18,2024-06-03
5,Standing Calf Raise (Machine),152.0,95.0,18,2025-12-20
6,Standing Calf Raise (Machine),152.0,95.0,18,2025-12-29
7,Standing Calf Raise (Machine),152.0,95.0,18,2025-12-29
8,Standing Calf Raise (Machine),152.0,95.0,18,2026-01-05
9,Standing Calf Raise (Machine),152.0,95.0,18,2026-01-05


### Q2 — Monthly Volume Trend

How did total training volume develop over the 6 years?

In [9]:
q2 = """
SELECT
    strftime('%Y-%m', date)        AS month,
    COUNT(DISTINCT workout_id)     AS workouts,
    ROUND(SUM(total_volume_kg), 0) AS total_volume_kg
FROM workouts
GROUP BY month
ORDER BY month;
"""
monthly = pd.read_sql(q2, conn)
monthly.tail(12)

,month,workouts,total_volume_kg
61,2025-06,16,137582.0
62,2025-07,13,125835.0
63,2025-08,8,60681.0
64,2025-09,4,22848.0
65,2025-10,9,83184.0
66,2025-11,1,6349.0
67,2025-12,9,90079.0
68,2026-01,9,72897.0
69,2026-02,11,136041.0
70,2026-03,6,71365.0


### Q3 — Most Trained Exercises

`workouts_performed` counts distinct sessions — more meaningful than raw set count.

In [10]:
q3 = """
SELECT
    exercise_name,
    COUNT(*)                     AS total_sets,
    COUNT(DISTINCT workout_id)   AS workouts_performed,
    ROUND(AVG(weight_kg), 1)     AS avg_weight_kg,
    ROUND(MAX(estimated_1rm), 1) AS best_1rm_kg
FROM sets
GROUP BY exercise_name
ORDER BY workouts_performed DESC
LIMIT 15;
"""
pd.read_sql(q3, conn)

,exercise_name,total_sets,workouts_performed,avg_weight_kg,best_1rm_kg
0,Lateral Raise (Dumbbell),726,210,7.9,18.4
1,Leg Extension (Machine),574,175,75.0,164.7
2,Incline Bench Press (Dumbbell),787,163,29.5,51.6
3,Cable Crossover,613,160,11.6,26.0
4,Lat Pulldown (Cable),599,155,74.1,138.0
5,Chest Dip,516,151,10.0,44.3
6,Bench Press (Barbell),753,150,63.6,114.7
7,Bicep Curl (Cable),486,138,11.3,42.2
8,Pull Up,559,134,7.1,27.9
9,Seated Leg Curl (Machine),467,133,60.5,120.8


### Q4 — Yearly Training Frequency

How consistent was the training across the 6 years?

In [11]:
q4 = """
SELECT
    strftime('%Y', date)           AS year,
    COUNT(*)                       AS total_workouts,
    ROUND(COUNT(*) / 52.0, 1)      AS avg_per_week,
    ROUND(AVG(duration_min), 1)    AS avg_duration_min,
    ROUND(AVG(total_volume_kg), 0) AS avg_volume_kg
FROM workouts
GROUP BY year
ORDER BY year;
"""
pd.read_sql(q4, conn)

,year,total_workouts,avg_per_week,avg_duration_min,avg_volume_kg
0,2020,122,2.3,71.8,7375.0
1,2021,191,3.7,75.3,8862.0
2,2022,155,3.0,72.2,13296.0
3,2023,90,1.7,73.3,15412.0
4,2024,126,2.4,77.5,13365.0
5,2025,117,2.3,84.6,9823.0
6,2026,48,0.9,76.5,10509.0


### Q5 — Strength Progression: Bench Press

Top 3 working sets per year by estimated 1RM. Change `exercise_name` to inspect any other lift.

In [ ]:
q5 = """
SELECT
    year,
    DATE(date)                     AS date,
    weight_kg,
    reps,
    ROUND(estimated_1rm, 1)        AS estimated_1rm_kg
FROM (
    SELECT
        strftime('%Y', date)       AS year,
        date,
        weight_kg,
        reps,
        estimated_1rm,
        ROW_NUMBER() OVER (
            PARTITION BY strftime('%Y', date)
            ORDER BY estimated_1rm DESC
        ) AS rn
    FROM sets
    WHERE exercise_name = 'Bench Press (Barbell)'
)
WHERE rn <= 3
ORDER BY year, rn;
"""
pd.read_sql(q5, conn)

In [ ]:
conn.close()
print(f"Database saved: {DB_PATH.resolve()}")